## This notebook can be used to rank a list of nodes from a category that connect to an entity such as a gene. 

In [2]:

import sys
import os
sys.path.append('../TCT/')
from TCT import node_normalizer
from TCT import name_resolver
from TCT import translator_metakg
from TCT import translator_kpinfo
from TCT import translator_query
from TCT import TCT_neighborhood_finder

from TCT import TCT



### Load Translator resources


In [3]:
APInames, metaKG, Translator_KP_info = translator_metakg.load_translator_resources()

Skipping server without x-maturity: {'url': '/sipr'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}


In [ ]:
#Translator_KP_info

In [4]:
All_predicates = list(set(metaKG['Predicate']))
All_categories = list((set(list(set(metaKG['Subject']))+list(set(metaKG['Object'])))))
API_withMetaKG = list(set(metaKG['API']))

# generate a dictionary of API and its predicates
API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(metaKG[metaKG['API'] == api]['Predicate']))

## Find the neighborhood of an entity from a subset of APIs 


In [5]:
# select a list of APIs to use and a list of predicates to use, if the list is empty, use all APIs and predicates
#selected_APIlist = ['Microbiome KP - TRAPI 1.5.0']
selected_APIlist = []
if len(selected_APIlist) == 0:
    select_APIs = APInames
else:
    select_APIs = {k: APInames[k] for k in selected_APIlist if k in APInames}

selected_metaKG = metaKG[metaKG['API'].isin(select_APIs.keys())]
print(select_APIs)
print(selected_metaKG.shape)


{'Clinical Trials KP - TRAPI 1.5.0': 'https://multiomics.rtx.ai:9990/ctkp/query', 'mediKanren': 'https://medikanren-trapi.transltr.io/query/', 'Microbiome KP - TRAPI 1.5.0': 'https://multiomics.rtx.ai:9990/mbkp/query', 'RTX KG2 - TRAPI 1.5.0': 'https://kg2cploverdb.ci.transltr.io/kg2c/query', 'Retriever': 'https://retriever.ci.transltr.io/query/', 'COHD TRAPI': 'https://cohd-api.transltr.io/api/query/', 'BioThings Explorer (BTE) TRAPI': 'https://bte.transltr.io/v1/query/', 'Automat-human-goa(Trapi v1.5.0)': 'https://automat.renci.org/human-goa/query/', 'ARAX Translator Reasoner - TRAPI 1.6.0': 'https://arax.transltr.io/api/arax/v1.4/query/', 'Multiomics KP - TRAPI 1.5.0': 'https://multiomics.rtx.ai:9990/multiomics/query', 'Genetics Data Provider for NCATS Biomedical Translator Reasoners': 'https://genetics-kp.transltr.io/genetics_provider/trapi/v1.5/query/', 'imProving Agent for TRAPI 1.5': 'https://ia.transltr.io/api/v1.5/query/', 'SPOKE KP for TRAPI 1.5': 'https://spokekp.transltr.io

In [8]:
name_resolver.lookup('TP53', only_taxa='NCBITaxon:9606', biolink_type='biolink:Gene')

TranslatorNode(curie='NCBIGene:7157', label='TP53', types=['biolink:Gene', 'biolink:GeneOrGeneProduct', 'biolink:GenomicEntity', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:PhysicalEssence', 'biolink:OntologyClass', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity', 'biolink:PhysicalEssenceOrOccurrent', 'biolink:MacromolecularMachineMixin', 'biolink:Protein', 'biolink:GeneProductMixin', 'biolink:Polypeptide', 'biolink:ChemicalEntityOrProteinOrPolypeptide'], synonyms=None, curie_synonyms=None, attributes=None, taxa=['NCBITaxon:9606'])

In [7]:
name_resolver.lookup('Acute Myeloid Leukemia', return_top_response=False, biolink_type='biolink:Disease',  limit=10) # sometimes the identifiers are not in the top 1, users need to check the other returned results


[TranslatorNode(curie='MONDO:0018874', label='acute myeloid leukemia', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]),
 TranslatorNode(curie='MONDO:0005223', label='acute myeloid leukemia with minimal differentiation', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]),
 TranslatorNode(curie='MONDO:0017893', label='inherited acute myeloid leukemia', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]),
 TranslatorNode(curie='MONDO:0020317', label='acute myeloid leukemia with 11q23 ab

In order to use the neighborhood finder, we have to look up a CURIE ID for a given term.

In [9]:
#name_resolver.lookup('BCL2')
#name_resolver.lookup('BCL2', return_top_response=False, biolink_type='biolink:Gene',  limit=100, only_taxa='NCBITaxon:9606') # sometimes the identifiers are not in the top 1, users need to check the other returned results

input_identifiers = 'MONDO:0018874'


Neighborhood finder identifies all nodes *b* that are connected to the given node *a*, where *b* is part of a defined list of categories - returning the neighborhood of node *a*.

In [14]:
input_node_id, result, result_parsed, result_ranked_by_primary_infores = TCT_neighborhood_finder.neighborhood_finder('MONDO:0018874',
                                                                                            node2_categories = ['biolink:Drug'],
                                                                                            APInames = select_APIs,
                                                                                            metaKG = selected_metaKG,
                                                                                            API_predicates = API_predicates)     

MONDO:0018874
Clinical Trials KP - TRAPI 1.5.0: Success!
RTX KG2 - TRAPI 1.5.0: Success!
Automat-robokop(Trapi v1.5.0): Success!
BioThings Explorer (BTE) TRAPI: Success!
NodeNorm does not know about these identifiers: UMLS:C5908001,UMLS:C5907931,UMLS:C5888788,UMLS:C5979854,UMLS:C5907992,DRUGBANK:DB15060,CHEBI:233362,RXCUI:1791496,RXCUI:1736582,RXCUI:1722942,RXCUI:1433769,RXCUI:1795154,RXCUI:1662281,RXCUI:1362058,RXCUI:1860240,RXCUI:1807509,RXCUI:1665191,RXCUI:1362049,RXCUI:1656667,RXCUI:1362054,RXCUI:1723777,RXCUI:379454,RXCUI:1860464,RXCUI:1658260,RXCUI:1795608,RXCUI:1666799,RXCUI:1729198,RXCUI:1360104,RXCUI:1719287,RXCUI:900963


In [15]:
TCT_path_finder_result = TCT_neighborhood_finder.parse_results_for_neighborhood_finder(input_identifiers, result,
        start_node_categories='biolink:Disease', end_node_categories=None,
        get_node_info=True,
        scoring_method='infores')

In [16]:
# write a result to a json file
import json
with open('_TestTCT_neighborhood_finder_result_'+input_identifiers.replace(':', '_')+'.json', 'w') as f:
    json.dump(TCT_path_finder_result, f)

In [ ]:
# End of the example
